# 13 — Pull NELDA Electoral Calendar Data

**Source:** National Elections Across Democracy and Autocracy (NELDA), v6  
**Provider:** Susan Hyde & Nikolay Marinov (2012)  
**Access:** Public download — Harvard Dataverse (DOI: 10.7910/DVN/IGFCHO)  
**Coverage:** Global, 1945–present (~200 countries, 4,000+ election events)  
**Frequency:** Election-event level; aggregated here to country-year  

## What this notebook pulls

NELDA codes every national election — executive, legislative, and local — along with
competitiveness, fraud allegations, and incumbent behaviour. The primary predictor
used by Medvedev et al. (2022) is a simple binary flag:

> **`elections_this_year`** — 1 if any national election was held in a country-year.  
> Coded as an *activating* predictor (positively associated with protest intensity)
> in both the ML model and the negative binomial regression cross-validation.
> Elections create heightened political mobilisation opportunities and periods of
> contested legitimacy that increase protest likelihood.

We also extract NELDA's richer election-quality variables for use in other sub-models.

### Election-event variables (raw NELDA)

| Variable | NELDA question | Description |
|---|---|---|
| `cow` / `ccode` | — | Correlates of War country code |
| `country` | — | Country name |
| `year` | — | Election year |
| `startdate` | — | Election start date |
| `eltype` | — | Election type: `exe` / `leg` / `both` / `local` |
| `nelda3` | More than one candidate/party? | Competitiveness indicator |
| `nelda5` | Significant irregularities reported? | Fraud/manipulation flag |
| `nelda14` | Did the incumbent run? | Incumbency flag |
| `nelda15` | Did the incumbent win? | Incumbency outcome |
| `nelda25` | International observers present? | Legitimacy indicator |
| `nelda26` | Were observers' reports positive? | Observer assessment |

### Country-year panel variables (derived)

| Variable | Description |
|---|---|
| `elections_this_year` | Any national election held (binary) — primary Medvedev predictor |
| `n_elections` | Count of elections in country-year |
| `exec_election` | Executive election held (binary) |
| `leg_election` | Legislative election held (binary) |
| `competitive_election` | Any election with >1 candidate/party (binary) |
| `fraud_allegations` | Any election with significant irregularities reported (binary) |
| `intl_observers` | International observers present at any election (binary) |
| `incumbent_ran` | Incumbent ran in any election (binary) |

## Access note
NELDA is freely available from Harvard Dataverse. Set `NELDA_URL` to the direct
file download URL from the Dataverse page, or place the file locally:

```
nelda_files/nelda.csv     ← CSV format (preferred)
nelda_files/nelda.dta     ← Stata format (fallback)
nelda_files/nelda.xlsx    ← Excel format (fallback)
```

Harvard Dataverse landing page: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/IGFCHO

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
NELDA_URL          — (optional) direct download URL from Harvard Dataverse
```

In [ ]:
import os
import io
import requests
import pandas as pd
import pyreadstat
from pathlib import Path
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Direct download URL from Harvard Dataverse — set after visiting the DOI landing page.
# Dataverse provides a stable API endpoint of the form:
#   https://dataverse.harvard.edu/api/access/datafile/<fileId>
# Copy the file ID from the Dataverse page for NELDA v6.
NELDA_URL = os.getenv("NELDA_URL", "")

# Local fallback paths
NELDA_LOCAL_DIR  = Path("nelda_files")
NELDA_LOCAL_CSV  = NELDA_LOCAL_DIR / "nelda.csv"
NELDA_LOCAL_DTA  = NELDA_LOCAL_DIR / "nelda.dta"
NELDA_LOCAL_XLSX = NELDA_LOCAL_DIR / "nelda.xlsx"

print(f"Run date        : {RUN_DATE}")
print(f"NELDA_URL set   : {bool(NELDA_URL)}")
print(
    f"Local files     : "
    f"csv={NELDA_LOCAL_CSV.exists()}, "
    f"dta={NELDA_LOCAL_DTA.exists()}, "
    f"xlsx={NELDA_LOCAL_XLSX.exists()}"
)

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Load NELDA data

Tries the download URL first, then searches `nelda_files/` for CSV → Stata → Excel.

In [ ]:
def _read_file_bytes(content: bytes, url_hint: str = "") -> pd.DataFrame | None:
    """Infer format from URL hint then content, returning a DataFrame."""
    hint = url_hint.lower()
    if hint.endswith(".dta"):
        df, _ = pyreadstat.read_dta(io.BytesIO(content))
        return df
    if hint.endswith((".xlsx", ".xls")):
        return pd.read_excel(io.BytesIO(content))
    # Default: try CSV first (most common Dataverse export), then Stata
    try:
        return pd.read_csv(io.BytesIO(content), low_memory=False)
    except Exception:
        pass
    try:
        df, _ = pyreadstat.read_dta(io.BytesIO(content))
        return df
    except Exception:
        return None


def load_nelda_from_url(url: str) -> pd.DataFrame | None:
    if not url:
        return None
    print(f"Downloading NELDA from URL...")
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
    except Exception as e:
        print(f"  Download failed: {e}")
        return None
    df = _read_file_bytes(resp.content, url_hint=url)
    if df is None:
        print("  Could not parse downloaded content as CSV, Stata, or Excel")
    return df


def load_nelda_from_local() -> pd.DataFrame | None:
    for path in [NELDA_LOCAL_CSV, NELDA_LOCAL_DTA, NELDA_LOCAL_XLSX]:
        if not path.exists():
            continue
        print(f"Loading NELDA from local file: {path}")
        try:
            if path.suffix == ".dta":
                df, _ = pyreadstat.read_dta(str(path))
            elif path.suffix in (".xlsx", ".xls"):
                df = pd.read_excel(str(path))
            else:
                df = pd.read_csv(str(path), low_memory=False)
            return df
        except Exception as e:
            print(f"  Failed ({e}); trying next format")
    return None


df_nelda_raw = load_nelda_from_url(NELDA_URL)

if df_nelda_raw is None:
    df_nelda_raw = load_nelda_from_local()

if df_nelda_raw is None:
    print(
        "\nWARNING: NELDA data not available.\n"
        "  Download NELDA v6 from Harvard Dataverse:\n"
        "  https://dataverse.harvard.edu/dataset.xhtml"
        "?persistentId=doi:10.7910/DVN/IGFCHO\n"
        "  Then either:\n"
        "  • Set env var NELDA_URL to the direct file download link, or\n"
        f"  • Place the file at {NELDA_LOCAL_CSV} (CSV) or {NELDA_LOCAL_DTA} (Stata)\n"
    )
else:
    df_nelda_raw.columns = [c.lower().strip() for c in df_nelda_raw.columns]
    print(
        f"NELDA raw: {len(df_nelda_raw):,} election events | "
        f"columns: {list(df_nelda_raw.columns)}"
    )

## Schema normalisation

NELDA column names are consistent across v5/v6 but casing and some variable names
shifted between releases. We map to a canonical schema and validate presence.

NELDA encodes yes/no questions as `"Yes"` / `"No"` strings (or `1`/`0` in some
exports). We normalise all such columns to `Int8` (1/0/NA).

In [ ]:
# Canonical column → possible source names (preference order)
COL_ALIASES = {
    "cow":       ["cow", "ccode", "cowcode", "countrycode"],
    "country":   ["country", "countryname", "country_name", "statename"],
    "year":      ["year"],
    "startdate": ["startdate", "start_date", "edate", "date"],
    "enddate":   ["enddate",   "end_date"],
    "eltype":    ["eltype", "el_type", "type", "election_type"],
    # Core NELDA yes/no questions used in political science literature
    "nelda3":    ["nelda3"],   # >1 candidate/party?
    "nelda5":    ["nelda5"],   # Significant irregularities reported?
    "nelda14":   ["nelda14"],  # Did the incumbent run?
    "nelda15":   ["nelda15"],  # Did the incumbent win?
    "nelda25":   ["nelda25"],  # International observers present?
    "nelda26":   ["nelda26"],  # Were observer reports positive?
}

# NELDA questions that map to our derived country-year flags
YESNO_COLS = ["nelda3", "nelda5", "nelda14", "nelda15", "nelda25", "nelda26"]

# Election type codes that count as executive or legislative
EXEC_TYPES = {"exe", "executive", "exec", "presidential", "both"}
LEG_TYPES  = {"leg", "legislative", "parliament", "parliamentary", "both"}


def yn_to_int(series: pd.Series) -> pd.Series:
    """Convert NELDA yes/no strings (or 1/0) to Int8."""
    s = series.astype(str).str.strip().str.lower()
    return (
        s.map({"yes": 1, "1": 1, "1.0": 1, "no": 0, "0": 0, "0.0": 0})
        .astype("Int8")
    )


if df_nelda_raw is not None:
    df_nelda = df_nelda_raw.copy()

    # Apply alias mapping
    rename_map = {}
    for canonical, aliases in COL_ALIASES.items():
        if canonical in df_nelda.columns:
            continue
        for alias in aliases:
            if alias in df_nelda.columns:
                rename_map[alias] = canonical
                break
    if rename_map:
        df_nelda.rename(columns=rename_map, inplace=True)
        print(f"Renamed columns: {rename_map}")

    # Parse dates
    for dcol in ["startdate", "enddate"]:
        if dcol in df_nelda.columns:
            df_nelda[dcol] = pd.to_datetime(df_nelda[dcol], errors="coerce")

    # Numeric identifiers
    for icol in ["cow", "year"]:
        if icol in df_nelda.columns:
            df_nelda[icol] = pd.to_numeric(df_nelda[icol], errors="coerce").astype("Int64")

    # Derive year from startdate when year column is absent
    if "year" not in df_nelda.columns and "startdate" in df_nelda.columns:
        df_nelda["year"] = df_nelda["startdate"].dt.year.astype("Int64")
        print("Derived 'year' from startdate")

    # Normalise yes/no columns to Int8
    for col in YESNO_COLS:
        if col in df_nelda.columns:
            df_nelda[col] = yn_to_int(df_nelda[col])

    # Normalise eltype to lowercase
    if "eltype" in df_nelda.columns:
        df_nelda["eltype"] = df_nelda["eltype"].astype(str).str.strip().str.lower()

    # Schema validation
    missing = [c for c in COL_ALIASES if c not in df_nelda.columns]
    if missing:
        print(f"WARNING: expected columns not found: {missing}")

    print(f"NELDA normalised: {len(df_nelda):,} events")
    if "year" in df_nelda.columns:
        print(f"Year range: {df_nelda['year'].min()}–{df_nelda['year'].max()}")
    df_nelda.head(3)
else:
    df_nelda = pd.DataFrame()

## Write election-event data to ADLS

In [ ]:
if not df_nelda.empty:
    write_parquet(df_nelda, f"raw/nelda/{RUN_DATE}/nelda_events.parquet")

## Derive country-year panel

The Medvedev et al. (2022) predictor is the binary `elections_this_year` flag.
We also carry forward the richer NELDA quality/competitiveness variables so that
other sub-models can use them without re-processing the event file.

Aggregation logic:
- **`elections_this_year`** — 1 if any election event exists in the country-year.
- **`n_elections`** — count of election events.
- **`exec_election`** / **`leg_election`** — 1 if any event of that type.
- **`competitive_election`** — 1 if `nelda3 == 1` for any event (>1 candidate/party).
- **`fraud_allegations`** — 1 if `nelda5 == 1` for any event.
- **`intl_observers`** — 1 if `nelda25 == 1` for any event.
- **`incumbent_ran`** — 1 if `nelda14 == 1` for any event.

In [ ]:
if not df_nelda.empty and "year" in df_nelda.columns:
    group_cols = [c for c in ["cow", "country", "year"] if c in df_nelda.columns]

    agg = df_nelda.groupby(group_cols, as_index=False)

    df_cy = agg.size().rename(columns={"size": "n_elections"})
    df_cy["elections_this_year"] = 1  # every group has ≥1 election by construction

    # Executive / legislative election flags (derived from eltype)
    if "eltype" in df_nelda.columns:
        exec_mask = df_nelda["eltype"].isin(EXEC_TYPES)
        leg_mask  = df_nelda["eltype"].isin(LEG_TYPES)

        exec_flag = (
            df_nelda[exec_mask]
            .groupby(group_cols, as_index=False)
            .size()
            .rename(columns={"size": "exec_election"})
        )
        exec_flag["exec_election"] = 1

        leg_flag = (
            df_nelda[leg_mask]
            .groupby(group_cols, as_index=False)
            .size()
            .rename(columns={"size": "leg_election"})
        )
        leg_flag["leg_election"] = 1

        df_cy = df_cy.merge(exec_flag, on=group_cols, how="left")
        df_cy = df_cy.merge(leg_flag,  on=group_cols, how="left")
        df_cy["exec_election"] = df_cy["exec_election"].fillna(0).astype("Int8")
        df_cy["leg_election"]  = df_cy["leg_election"].fillna(0).astype("Int8")

    # Yes/no question flags — 1 if *any* election in the country-year answered "Yes"
    yn_flag_map = {
        "nelda3":  "competitive_election",
        "nelda5":  "fraud_allegations",
        "nelda14": "incumbent_ran",
        "nelda15": "incumbent_won",
        "nelda25": "intl_observers",
        "nelda26": "observers_positive",
    }
    for nelda_col, derived_col in yn_flag_map.items():
        if nelda_col not in df_nelda.columns:
            continue
        flag = (
            df_nelda.groupby(group_cols, as_index=False)[nelda_col]
            .max()  # max of Int8: 1 if any event == 1
            .rename(columns={nelda_col: derived_col})
        )
        df_cy = df_cy.merge(flag, on=group_cols, how="left")
        df_cy[derived_col] = df_cy[derived_col].astype("Int8")

    # Cast identifier columns
    df_cy["elections_this_year"] = df_cy["elections_this_year"].astype("Int8")
    df_cy["n_elections"]         = df_cy["n_elections"].astype("Int64")
    if "cow" in df_cy.columns:
        df_cy["cow"] = df_cy["cow"].astype("Int64")
    if "year" in df_cy.columns:
        df_cy["year"] = df_cy["year"].astype("Int64")

    print(f"Country-year panel: {len(df_cy):,} rows")
    print(f"Years     : {df_cy['year'].min()}–{df_cy['year'].max()}")
    if "country" in df_cy.columns:
        print(f"Countries : {df_cy['country'].nunique()}")
    print(f"Columns   : {list(df_cy.columns)}")
    df_cy.head(3)
else:
    df_cy = pd.DataFrame()

In [ ]:
if not df_cy.empty:
    write_parquet(df_cy, f"raw/nelda/{RUN_DATE}/nelda_country_year.parquet")

## Sanity check — elections_this_year coverage

Compare annual election counts against known patterns:
most country-years should have no election; ~20–30% should have one.

In [ ]:
if not df_cy.empty and "year" in df_cy.columns:
    # Annual count of countries holding at least one election
    annual = (
        df_cy
        .groupby("year", as_index=False)["elections_this_year"]
        .sum()
        .rename(columns={"elections_this_year": "countries_with_election"})
    )
    print("Annual countries with ≥1 election (sample):")
    print(annual.tail(20).to_string(index=False))

    # Overall share
    if "country" in df_cy.columns:
        all_cy_count = df_cy["year"].nunique() * df_cy["country"].nunique()
        election_rate = len(df_cy) / all_cy_count if all_cy_count else float("nan")
        print(f"\nElection events represent {election_rate:.1%} of all observed country-years")

## Summary

In [ ]:
print("=" * 55)
print("NELDA pull complete")
print("=" * 55)
if not df_nelda.empty:
    print(f"  Election events   : {len(df_nelda):,}")
if not df_cy.empty:
    print(f"  Country-years     : {len(df_cy):,}")
    print(f"  Year range        : {df_cy['year'].min()}–{df_cy['year'].max()}")
    if "country" in df_cy.columns:
        print(f"  Countries         : {df_cy['country'].nunique()}")
    if "elections_this_year" in df_cy.columns:
        n_election_years = int(df_cy["elections_this_year"].sum())
        print(f"  Country-yrs w/ election: {n_election_years:,}")
else:
    print("  No data loaded — see WARNING above for access instructions.")
print()
print("ADLS paths written:")
print(f"  raw/nelda/{RUN_DATE}/nelda_events.parquet")
print(f"  raw/nelda/{RUN_DATE}/nelda_country_year.parquet")